<a href="https://colab.research.google.com/github/SergioCuadrado08/Teoria_Aprendizaje_Maquinas_2025-2/blob/main/SergioCuadrado_TAM_IntroductionDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Librerias y carga de datos

In [12]:
from IPython.display import clear_output
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
import numpy as np
import pandas as pd
import time

In [13]:
# Cargar Fashion Mnist
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

X_valid, X_train = X_train[:5000] / 255., X_train[5000:] / 255.
y_valid, y_train = y_train[:5000], y_train[5000:]
X_test = X_test / 255.

clear_output()
print("Train:", X_train.shape); print("Valid:", X_valid.shape);print("Test:", X_test.shape)

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat","Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

Train: (55000, 28, 28)
Valid: (5000, 28, 28)
Test: (10000, 28, 28)


# Ejercicio

Configure el entorno de Colab para trabajar con GPU. Repita el entrenamiento del modelo de clasificación Fashion mnist para batch size en 32, 64, 128, 256, y 512. Reporte una tabla con los rendimientos sobre el conjunto de test y el tiempo promedio de cómputo por época. (Repita el procedimiento del punto anterior configurando Colab para trabajar con TPU.)

In [ ]:
batch_sizes = [32, 64, 128, 256, 512]
results = []
histories = {}

for batch in batch_sizes:
    # Definir un nuevo modelo secuencial
    model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(300, activation="relu"),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    # Compilar el modelo
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="sgd",
        metrics=["accuracy"]
    )

    # Medir el tiempo de entrenamiento
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=batch,
        validation_data=(X_valid, y_valid),
        verbose=0
    )
    end_time = time.time()

    # Calcular el tiempo promedio por época
    avg_time = (end_time - start_time) / 10

    # Evaluar el modelo con el conjunto de prueba
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    # Almacenar los resultados
    results.append({
        "Batch Size": batch,
        "Test Accuracy": test_acc,
        "Test Loss": test_loss,
        "Avg Time per Epoch (s)": avg_time
    })

    histories[batch] = history.history

    clear_output(wait=True)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Tabla de Resultados
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Gráfica de la pérdida durante el entrenamiento
plt.figure(figsize=(16, 5))
for batch, history in histories.items():
    plt.plot(history["loss"], label=f"Batch = {batch}")
plt.title("Evolución de la pérdida por época")
plt.xlabel("Época")
plt.ylabel("Pérdida (Loss)")
plt.legend()
plt.grid(True)
plt.show()
print("")

# Gráfica de la precisión durante el entrenamiento
plt.figure(figsize=(16, 5))
for batch, history in histories.items():
    plt.plot(history["accuracy"], label=f"Batch = {batch}")
plt.title("Evolución de la precisión por época")
plt.xlabel("Época")
plt.ylabel("Precisión (Accuracy)")
plt.legend()
plt.grid(True)
plt.show()


## Con TPU

In [ ]:
!pip install tensorflow tensorflow_gcs_config tensorflow_datasets -q
clear_output()

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
# Cargar Fashion Mnist
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

X_valid, X_train = X_train[:5000] / 255., X_train[5000:] / 255.
y_valid, y_train = y_train[:5000], y_train[5000:]
X_test = X_test / 255.

clear_output()
print("Train:", X_train.shape); print("Valid:", X_valid.shape);print("Test:", X_test.shape)

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat","Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
batch_sizes = [32, 64, 128, 256, 512]
results = []
histories = {}

for batch in batch_sizes:
    model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(300, activation="relu"),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer="sgd", metrics=["accuracy"])

    start_time = time.time()
    history = model.fit(X_train, y_train, epochs=10, batch_size=batch,
                        validation_data=(X_valid, y_valid), verbose=0)
    end_time = time.time()

    avg_time = (end_time - start_time) / 10
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    results.append({
        "Batch Size": batch,
        "Test Accuracy": test_acc,
        "Test Loss": test_loss,
        "Avg Time per Epoch (s)": avg_time
    })
    histories[batch] = history.history

    clear_output(wait=True)


In [ ]:
# Tabla de Resultados
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Pérdida durante el entrenamiento
plt.figure(figsize=(16, 5))
for batch, hist in histories.items():
    plt.plot(hist["loss"], label=f"Batch = {batch}")
plt.title("Evolución de la pérdida por época")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.show()
print()

# Precisión durante el entrenamiento
plt.figure(figsize=(16, 5))
for batch, hist in histories.items():
    plt.plot(hist["accuracy"], label=f"Batch = {batch}")
plt.title("Evolución de la precisión por época")
plt.xlabel("Época")
plt.ylabel("Precisión")
plt.legend()
plt.grid(True)
plt.show()